# HIT-ASR — main results on AMI-SDM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huseyin-karaca/hit-asr/blob/main/notebooks/main_ami_sdm.ipynb)

This notebook reproduces the main experiment of the paper on AMI-SDM. Every model's hyperparameters are selected on a
25 % hold-out of the corpus; every model is then trained and tested on the same ten folds of a 5×2 cross-validation
of the remaining clips, and HIT-ASR is compared with every other system on those folds.

The configuration is fixed in `hitasr.configs.MAIN["ami_sdm"]` and the code is in `hitasr.experiment`. The data and the
stored searches and folds are read from the Hugging Face dataset [`huseyin-karaca/hit-asr`](https://huggingface.co/datasets/huseyin-karaca/hit-asr); nothing is written to it.

| Level | What runs | Needs |
|---|---|---|
| **1** | every table, from the searches and folds stored on the Hub | a CPU, minutes |
| **2** | every fold retrained from the published labels and frames (`RETUNE = True`: the search as well) | a GPU, hours |
| **3** | level 2 on labels and frames rebuilt from the audio: run `extract` first on the same machine, which writes them to a local folder that level 3 reads (`HITASR_HUB` names another place) | a GPU |

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("hitasr") is None:     # Colab: install the package (a local or Docker run already has it)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "hit-asr[paper] @ git+https://github.com/huseyin-karaca/hit-asr.git@main"], check=True)

In [2]:
from hitasr.configs import MAIN
from hitasr.experiment import MainExperiment

LEVEL = 1          # 1, 2 or 3 (see above)
RETUNE = False     # levels 2 and 3: run the hyperparameter search again
DEVICE = "cuda"    # levels 2 and 3 train on it

exp = MainExperiment(MAIN["ami_sdm"], level=LEVEL, retune=RETUNE, device=DEVICE)

AMI-SDM: level 1 — search read from the Hub, folds read from the Hub
experts  Cohere-Transcribe, Granite-Speech-4.1-2b, Parakeet-TDT-0.6b-v3
arms     HIT-ASR, MLP-pool, ADASTT, ROVER (confidence-weighted), CN-MBR, MLP-pool (hard CE)


## 1. Hyperparameter spaces (Table 2)

The free hyperparameters of every model and their choices; every other parameter is fixed
(`hitasr.configs.SPACES`).

In [3]:
exp.spaces()

Table 2 — hyperparameter spaces
10 random draws per arm; parameters not listed are fixed (hitasr.configs.SPACES)
  [HIT-ASR  (16 free of 20)]
      parameter                                     choices
        d_model                                  {128, 256}
        n_heads                                      {4, 8}
  stage1_layers                                   {1, 2, 3}
  stage2_layers                                      {1, 2}
        ffn_dim                            {256, 512, 1024}
        dropout                                {0.05, 0.15}
         fusion           {bridge, self_attn, concat, mean}
   share_stage1                               {True, False}
     input_norm                  {none, scale, standardize}
             lr                                    {0.0001}
   weight_decay                          {0.001, 0.01, 0.1}
     batch_size                             {8, 16, 32, 64}
     max_frames                           {500, 1000, 2000}
       patience                                     {5, 10}
    loss_preset {hard_ce_only, soft_ce_only, wer+hard, all}
label_smoothing                                  {0.0, 0.1}
  [MLP-pool  (8 free of 12)]
      parameter                        choices
       d_hidden         {256, 512, 1024, 2048}
       n_layers                      {1, 2, 3}
        dropout              {0.05, 0.15, 0.3}
             lr {3e-05, 0.0001, 0.0003, 0.001}
   weight_decay             {0.001, 0.01, 0.1}
     batch_size             {32, 64, 128, 256}
       patience                        {5, 10}
label_smoothing                     {0.0, 0.1}
  [ADASTT  (7 free of 8)]
       parameter                choices
    n_estimators       {100, 400, 1000}
   learning_rate {0.01, 0.03, 0.1, 0.3}
       max_depth              {3, 5, 8}
       subsample        {0.6, 0.8, 1.0}
colsample_bytree   {0.1, 0.3, 0.6, 1.0}
min_child_weight       {1.0, 4.0, 16.0}
      reg_lambda       {0.3, 3.0, 30.0}
  [ROVER (confidence-weighted)  (6 free of 6)]
     parameter                          choices
         alpha                            {0.0}
   conf_source               {cn, probe, blend}
     weighting {uniform, inverse_wer, log_odds}
weight_by_conf                    {False, True}
     conf_temp                            {2.0}
  null_penalty                      {0.25, 0.5}
  [CN-MBR  (7 free of 7)]
  parameter                          choices
     decode                      {slot, hyp}
 candidates              {members+consensus}
      prior {uniform, inverse_wer, log_odds}
temperature                           {10.0}
 null_prior                      {0.05, 0.2}
   use_conf                    {False, True}
conf_source                          {probe}
  [MLP-pool (hard CE)  (8 free of 12)]
      parameter                        choices
       d_hidden         {256, 512, 1024, 2048}
       n_layers                      {1, 2, 3}
        dropout              {0.05, 0.15, 0.3}
             lr {3e-05, 0.0001, 0.0003, 0.001}
   weight_decay             {0.001, 0.01, 0.1}
     batch_size             {32, 64, 128, 256}
       patience                        {5, 10}
label_smoothing                     {0.0, 0.1}

## 2. Data

The three experts' transcripts and word-error counters for every clip (at levels 2 and 3, also their frame-level
encoder states). The stored counters are recomputed from the stored transcripts as a check.

In [4]:
exp.load()

cohere_transcribe: 1 of 41,641 pooled vectors are non-finite -> zeroed


parakeet_tdt_0_6b_v3: 1 of 41,641 pooled vectors are non-finite -> zeroed


ami_sdm: 41,641 utterances x 3 experts (0 adopted from fastt) in 3.6s
100.0% of 1,200 (utterance, model) counters reproduce exactly


RouterStore(ami_sdm, cohere_transcribe+granite_speech_4_1_2b+parakeet_tdt_0_6b_v3, 41,641 rows, frames closed)

## 3. Hyperparameter search

A random 25 % of the clips is held out for the search. For every model, ten configurations are drawn at random from
its space (Table 2), each trained on 70 % of the hold-out and scored on the remaining 30 %; the configuration with the
lowest corpus WER is kept. The other 75 % of the clips are never seen by the search.

In [5]:
exp.search()

partition,n_utts,share
tune,10410,0.250
eval,31231,0.750
fit,7287,0.175
val,3123,0.075


  on the tuning-score rows: bsm(cohere_transcribe) 0.1881   oracle 0.1213   gap 0.0668
  the 31,231 evaluation rows are never seen by a trial
  HIT-ASR 10 trials; MLP-pool 10 trials; ADASTT 10 trials; ROVER (confidence-weighted) 10 trials; CN-MBR 10 trials; MLP-pool (hard CE) 10 trials — read from the Hub


HPTRun('hpt_ami_sdm_e1917058', 6 arm(s), best hit_asr 0.1729, eval on 31,231 rows)

In [6]:
exp.dataset()

statistic,ami_sdm
Total clips,41641
Hyperparameter hold-out,10410
Evaluation rows (5x2),31231
Train size (per fold),15615
Test size (per fold),15616
Hours,29.6620
Mean clip length (s),2.5644
Reference words,315864


In [7]:
exp.selected()

label,trials,hold-out WER,selected configuration
HIT-ASR,10,0.1729,"{""batch_size"": 8, ""d_model"": 256, ""dropout"": 0.05, ""ffn_dim"": 1024, ""fusion"": ""concat"", ""input_norm"": ""none"", ""label_smoothing"": 0.1, ""loss_preset"": ""soft_ce_only"", ""lr"": 0.0001, ""max_frames"": 2000, ""max_restarts"": 0, ""n_heads"": 8, ""n_seeds"": 3, ""patience"": 5, ""pooled_skip"": false, ""share_stage1"": false, ""stage1_layers"": 3, ""stage2_layers"": 2, ""weight_decay"": 0.1, ""word_weighted"": false}"
MLP-pool,10,0.1859,"{""batch_size"": 64, ""d_hidden"": 2048, ""dropout"": 0.05, ""label_smoothing"": 0.1, ""loss_preset"": ""all"", ""lr"": 3e-05, ""max_restarts"": 0, ""n_layers"": 3, ""n_seeds"": 3, ""patience"": 10, ""weight_decay"": 0.01, ""word_weighted"": false}"
ADASTT,10,0.1872,"{""colsample_bytree"": 0.1, ""learning_rate"": 0.3, ""max_depth"": 8, ""min_child_weight"": 16.0, ""n_estimators"": 400, ""objective"": ""cross_entropy"", ""reg_lambda"": 3.0, ""subsample"": 0.8}"
ROVER (confidence-weighted),10,0.1985,"{""alpha"": 0.0, ""conf_source"": ""blend"", ""conf_temp"": 2.0, ""null_penalty"": 0.5, ""weight_by_conf"": true, ""weighting"": ""log_odds""}"
CN-MBR,10,0.2434,"{""candidates"": ""members+consensus"", ""conf_source"": ""probe"", ""decode"": ""hyp"", ""null_prior"": 0.05, ""prior"": ""log_odds"", ""temperature"": 10.0, ""use_conf"": true}"
MLP-pool (hard CE),10,0.1909,"{""batch_size"": 64, ""d_hidden"": 1024, ""dropout"": 0.05, ""label_smoothing"": 0.1, ""loss_preset"": ""hard_ce_only"", ""lr"": 0.001, ""max_restarts"": 0, ""n_layers"": 3, ""n_seeds"": 3, ""patience"": 5, ""weight_decay"": 0.01, ""word_weighted"": false}"


## 4. 5×2 cross-validation

Five repetitions, each splitting the evaluation clips into two halves with its own seed. Every model is trained on
one half and tested on the other, then the other way round: ten folds, the same for every model.

In [8]:
exp.cross_validate();

  hub: runlog main_ami_sdm — 35 of 35 (seed, group) result(s) available; fetched 70 file(s) in 1s
  runlog HIT  cv5x2 seed 17 — 7/7 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 11 — 7/7 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 22 — 7/7 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 21 — 7/7 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 99 — 7/7 group(s) cached, nothing refitted


5 repetitions x 2 folds x 15 arms


## 5. Main results (Table 3)

In [9]:
exp.results()

Table 3 — main results, AMI-SDM
mean ± sd over 10 folds; WER and uWER (mean utterance WER) as fractions, GC (share of the gap between the best single expert and the oracle closed) and Acc (selection accuracy) in %; dist = share of clips routed to Cohere-Transcribe / Granite-Speech-4.1-2b / Parakeet-TDT-0.6b-v3
  [expert]
                                   arm             WER            uWER         GC@0      Acc@0           dist
                     Cohere-Transcribe 0.1842 ± 0.0022 0.4654 ± 0.0035    0.0 ± 0.0 73.6 ± 0.4 1.00-0.00-0.00
                 Granite-Speech-4.1-2b 0.1957 ± 0.0049 0.3833 ± 0.0092  -17.0 ± 7.5 76.5 ± 0.3 0.00-1.00-0.00
                  Parakeet-TDT-0.6b-v3 0.2623 ± 0.0025 0.5347 ± 0.0028 -115.8 ± 4.1 56.3 ± 0.4 0.00-0.00-1.00
Best single expert (Cohere-Transcribe) 0.1842 ± 0.0022 0.4654 ± 0.0035    0.0 ± 0.0 73.6 ± 0.4 1.00-0.00-0.00
  [reference]
   arm             WER            uWER        GC@0      Acc@0           dist
Random 0.2142 ± 0.0048 0.4609 ± 0.0078 -44.5 ± 7.0 68.8 ± 0.3 0.33-0.33-0.33
  [fusion]
                        arm             WER            uWER        GC@0 Acc@0 dist
ROVER (confidence-weighted) 0.2003 ± 0.0037 0.4445 ± 0.0092 -23.8 ± 4.4     —    —
                     CN-MBR 0.2399 ± 0.0044 0.5863 ± 0.0078 -82.7 ± 6.0     —    —
  [pooled]
     arm             WER            uWER       GC@0      Acc@0           dist
MLP-pool 0.1744 ± 0.0021 0.4002 ± 0.0051 14.5 ± 3.6 78.4 ± 0.5 0.75-0.21-0.04
  ADASTT 0.1823 ± 0.0022 0.4493 ± 0.0046  2.9 ± 2.0 74.9 ± 0.4 0.96-0.04-0.00
  [ours]
    arm             WER            uWER       GC@0      Acc@0           dist
HIT-ASR 0.1719 ± 0.0030 0.3749 ± 0.0068 18.2 ± 5.7 79.7 ± 0.6 0.40-0.54-0.06
  [oracle]
                      arm             WER            uWER        GC@0       Acc@0           dist
                   Oracle 0.1168 ± 0.0013 0.2470 ± 0.0028 100.0 ± 0.0 100.0 ± 0.0 0.74-0.21-0.06
Oracle (1-word tolerance) 0.1367 ± 0.0015 0.3227 ± 0.0031  70.4 ± 0.5  84.8 ± 0.3 0.89-0.09-0.03
Oracle (2-word tolerance) 0.1515 ± 0.0015 0.3690 ± 0.0027  48.5 ± 0.6  79.2 ± 0.3 0.94-0.04-0.01

In [10]:
exp.routing()

router,Cohere-Transcribe,Granite-Speech-4.1-2b,Parakeet-TDT-0.6b-v3,entropy,switch rate
Random,33.4,33.3,33.3,1.585,66.6
MLP-pool,74.8,21.0,4.3,0.980,25.2
ADASTT,96.1,3.8,0.1,0.241,3.9
HIT-ASR,39.7,54.1,6.2,1.257,60.3
Oracle,73.6,20.8,5.6,1.029,26.4
Oracle (1-word tolerance),88.7,8.7,2.5,0.594,11.3
Oracle (2-word tolerance),94.4,4.3,1.3,0.357,5.6


## 6. Statistical test (Table 6)

HIT-ASR against every other system on the same ten folds, with a custom statistical test. For full details, please
see the paper.

In [11]:
exp.tests()

Table 6 — HIT-ASR against every system, AMI-SDM
on the same 10 folds; effect in the metric's own units
anchor: HIT-ASR; positive effect = anchor better
  [corpus WER]
                 comparator  effect       t      p
          Cohere-Transcribe +0.0123   4.315 0.0076
      Granite-Speech-4.1-2b +0.0238   4.554 0.0061
       Parakeet-TDT-0.6b-v3 +0.0903  22.729 0.0000
         Best single expert +0.0123   4.315 0.0076
                     Random +0.0423  13.047 0.0000
ROVER (confidence-weighted) +0.0283   7.674 0.0006
                     CN-MBR +0.0680  16.332 0.0000
                   MLP-pool +0.0025   2.889 0.0342
                     ADASTT +0.0103   4.034 0.0100
                     Oracle -0.0551 -13.502 0.0000
  Oracle (1-word tolerance) -0.0352  -7.981 0.0005
  Oracle (2-word tolerance) -0.0204  -4.053 0.0098
  [mean utterance WER]
                 comparator  effect       t      p
          Cohere-Transcribe +0.0905  10.931 0.0001
      Granite-Speech-4.1-2b +0.0084   0.576 0.5893
       Parakeet-TDT-0.6b-v3 +0.1598  17.605 0.0000
         Best single expert +0.0905  10.931 0.0001
                     Random +0.0860  11.787 0.0001
ROVER (confidence-weighted) +0.0696   5.972 0.0019
                     CN-MBR +0.2114  21.114 0.0000
                   MLP-pool +0.0253   8.220 0.0004
                     ADASTT +0.0744   8.745 0.0003
                     Oracle -0.1279 -13.580 0.0000
  Oracle (1-word tolerance) -0.0523  -4.690 0.0054
  Oracle (2-word tolerance) -0.0060   0.366 0.7292
Custom statistical test — for full details, please see the paper.

## 7. Record

The configuration, the selected hyperparameters and every table above, in `main_ami_sdm.json`.

In [12]:
exp.save();

wrote main_ami_sdm.json
